In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib.pyplot as plt
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import pdist
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, Conv1D, BatchNormalization, Bidirectional, LSTM, Concatenate, Activation, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber, MeanSquaredError, LogCosh, MeanAbsoluteError
from tensorflow.keras.initializers import GlorotUniform, Orthogonal
import json
from scipy.stats import zscore
from utils import random_extraction, extract_hrv_features, fast_predict, evaluate_imputation_performance

I0000 00:00:1785779626.046580 2495398 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785779629.369132 2495398 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# Cargar los datos
series_list = 'series/'
files_test = [
    "4016.txt",
    "4005.txt",
    "4062.txt",
    "4099.txt",
    "4012.txt",
    "4085.txt",
    "4086.txt",
    "403.txt",
    "16786.txt",
    "nsr010RRcl.txt",
    "nsr003RRcl.txt"
]
window_size = 20
np.random.seed(7)
SEED = 7

feature_cols = ['mean', 'sdsd', 'sd2', 'ccm', 'guzik', 'nn50', 'porta', 'std']
rr_cols = [f'rr_{i}' for i in range(1, 21)]
rr_last_col = 'rr_20'

model_path = 'cnn_lstm_hrv_generalist.keras'
# Load the Keras Model
print("Loading model...")
loaded_model = load_model(model_path)
print("✅ Model loaded successfully.")

ages_table_path = 'ages_table.xlsx'
ages_table = pd.read_excel(ages_table_path)

Loading model...


E0000 00:00:1785779639.865352 2495398 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1785779639.866346 2495592 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
E0000 00:00:1785779639.880065 2495398 cuda_executor.cc:1827] Nvml call failed with 3(Not Supported). Assuming PCIe gen 3 x16 bandwidth.
W0000 00:00:1785779639.882105 2495398 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


✅ Model loaded successfully.


In [3]:
import h5py
import numpy as np


feature_cols = [
    "mean",
    "sdsd",
    "sd2",
    "ccm",
    "guzik",
    "nn50",
    "porta",
    "std",
]

rr_cols = [f"rr_{i}" for i in range(1, 21)]


def get_rr_reference(age_years):

    seq_mean = 505 * age_years**0.122

    if age_years <= 12:
        seq_scale = 80 * age_years**0.26
    else:
        seq_scale = 290 * age_years**(-0.2)

    return float(seq_mean), float(seq_scale)


def canonical_code(x):
    s = str(x).strip()
    if s.isdigit():
        return f"{int(s):03d}" if len(s) < 3 else s
    return s

def get_subject_interval(h5_path, subject):
    subject_key = canonical_code(subject)

    with h5py.File(h5_path, "r") as h5:
        subject_ids = [
            x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x)
            for x in h5["index"]["subject_id"][()]
        ]
        intervals = [
            x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x)
            for x in h5["index"]["interval"][()]
        ]

        for sid, interval in zip(subject_ids, intervals):
            if canonical_code(sid) == subject_key:
                return interval

    raise ValueError(f"Subject {subject} not found in {h5_path}")

def load_normalization_params(
    h5_path,
    interval,
    age_years,
):
    """
    Returns:
        feats_mean : (8,)
        feats_scale : (8,)
        seq_mean : scalar
        seq_scale : scalar
        y_mean : scalar
        y_scale : scalar
    """

    with h5py.File(h5_path, "r") as h5:

        grp = h5["normalization"][str(interval)]

        cols = [
            c.decode()
            if isinstance(c, bytes)
            else str(c)
            for c in grp["columns"][:]
        ]

        mean = grp["mean"][:]
        std = grp["std"][:]

    stats = {
        col: (m, s)
        for col, m, s in zip(cols, mean, std)
    }

    # ----------------------------------
    # Static features
    # ----------------------------------

    feats_mean = np.array([
        get_rr_reference(age_years)[0],  # mean
        stats["sdsd"][0],
        stats["sd2"][0],
        stats["ccm"][0],
        stats["guzik"][0],
        stats["nn50"][0],
        stats["porta"][0],
        stats["std"][0],
    ], dtype=np.float32)

    feats_scale = np.array([
        get_rr_reference(age_years)[1],  # mean scale
        stats["sdsd"][1],
        stats["sd2"][1],
        stats["ccm"][1],
        stats["guzik"][1],
        stats["nn50"][1],
        stats["porta"][1],
        stats["std"][1],
    ], dtype=np.float32)

    # ----------------------------------
    # RR sequence
    # ----------------------------------

    seq_mean, seq_scale = get_rr_reference(age_years)

    # ----------------------------------
    # Target
    # ----------------------------------

    y_mean = float(stats["target"][0])
    y_scale = float(stats["target"][1])

    return (
        feats_mean,
        feats_scale,
        seq_mean,
        seq_scale,
        y_mean,
        y_scale,
    )

In [4]:
interval = get_subject_interval("hrv_test.h5", "4016")
print(interval)
age_weeks = ages_table['age-weeks'].loc[ages_table['code'] == 4016].values[0]
age_years = age_weeks / 52.14

(
    feats_mean,
    feats_scale,
    seq_mean,
    seq_scale,
    y_mean,
    y_scale,
) = load_normalization_params(
    "hrv_dataset.h5",
    interval,
    age_years,
)

46-52


In [5]:
# Create your array of percentages
file = files_test[0]
percents_array = np.arange(0.01, 0.85, 0.05)
original_serie = np.loadtxt(os.path.join(series_list, file), dtype=int).astype(float)
print(os.path.join(series_list, file))



# Run the function
rmse_results, mae_results, r2_results, corr_results = evaluate_imputation_performance(
    original_serie=original_serie,
    percents_to_eliminate=percents_array,
    loaded_model=loaded_model,
    feats_mean=feats_mean, feats_scale=feats_scale,
    seq_mean=seq_mean, seq_scale=seq_scale,
    y_mean=y_mean, y_scale=y_scale,
    feature_cols=feature_cols, rr_cols=rr_cols
)
subject_name = os.path.splitext(file)[0]

# 2. Combine the arrays into a structured Pandas DataFrame
# (Assuming 'percents_array' is the array you used for the loop)
results_df = pd.DataFrame({
    'Percent_Eliminated': percents_array,
    'RMSE': rmse_results,
    'MAE': mae_results,
    'R2': r2_results,
    'Correlation': corr_results
})
    
results_df.to_csv(f'imputation_metrics_{subject_name}.csv', index=False)

series/4016.txt
Starting autoregressive imputation across 17 thresholds...
Evaluating elimination: 1.00%
Evaluating elimination: 6.00%
Evaluating elimination: 11.00%
Evaluating elimination: 16.00%
Evaluating elimination: 21.00%
Evaluating elimination: 26.00%
Evaluating elimination: 31.00%
Evaluating elimination: 36.00%
Evaluating elimination: 41.00%
Evaluating elimination: 46.00%
Evaluating elimination: 51.00%
Evaluating elimination: 56.00%
Evaluating elimination: 61.00%
Evaluating elimination: 66.00%
Evaluating elimination: 71.00%
Evaluating elimination: 76.00%
Evaluating elimination: 81.00%
✅ All thresholds evaluated successfully!
